# 03 — VLM Analysis Preview

In [ ]:
# Cell 0: Health check
import json
from pathlib import Path
from collections import Counter

REPO_ROOT = Path('../')
ANALYSIS_REGISTRY = REPO_ROOT / 'data/analysis/analysis_registry.json'

with open(ANALYSIS_REGISTRY) as f:
    master = json.load(f)

results = master['results']

OTTOMAN_TERMS = [
    'ottoman', 'mosque', 'pointed arch', 'islamic', 'minaret',
    'hacı bayram', 'haci bayram', 'geometric ornament', 'byzantine', 'medieval',
]

# Pass type breakdown
pass_counts = Counter(r.get('pass', 'unknown') for r in results)
print('Pass type breakdown:')
for k, v in sorted(pass_counts.items()):
    print(f'  {k:<20} {v}')
print()

# JSON parse failures
failures = [r for r in results if 'error' in r]
print(f'JSON parse failures: {len(failures)}')
for r in failures:
    print(f'  {r["source_filename"]} — {r.get("error", "?")[:80]}')
print()

# Mosque interference
mosque_counts = Counter(r.get('mosque_interference', 'unknown') for r in results)
print('Mosque interference:')
for k, v in sorted(mosque_counts.items()):
    print(f'  {k:<12} {v}')
print()

# SDXL Ottoman contamination check
contaminated = [
    r['source_filename'] for r in results
    if any(t in (r.get('sdxl_prompt_component') or '').lower() for t in OTTOMAN_TERMS)
]
print(f'SDXL contamination (Ottoman terms remaining): {len(contaminated)}')
for f in contaminated:
    print(f'  {f}')
print()

# Truncation check (missing stratigraphic_confidence)
truncated = [
    r['source_filename'] for r in results
    if 'error' not in r and not r.get('stratigraphic_confidence')
]
print(f'Truncated outputs (missing stratigraphic_confidence): {len(truncated)}')
for f in truncated:
    print(f'  {f}')

In [ ]:
# Cell 1: Summary stats
summary = master['summary']

print(f"Total analyzed : {summary['total_analyzed']}")
print(f"Succeeded      : {summary.get('succeeded', '?')}")
print(f"Failed         : {summary.get('failed', '?')}")
print()
print('Roman fabric quality:')
for k, v in sorted(summary['by_roman_fabric_quality'].items()):
    print(f'  {k:<12} {v}')
print()
print('Credibility tier:')
for k, v in sorted(summary['by_credibility_tier'].items()):
    print(f'  {k:<12} {v}')

In [ ]:
# Cell 2: Two-pass result — before/after RAG
two_pass = next((r for r in results if r.get('pass') == 'two_pass'), None)

if two_pass is None:
    print('No two-pass results found yet.')
else:
    print(f"Image   : {two_pass['source_filename']}")
    print(f"Folder  : {two_pass['source_folder']}")
    print(f"Tier    : {two_pass['credibility_tier']}")
    print()
    print('Retrieved citations:')
    for c in two_pass.get('retrieved_citations', []):
        print(f'  - {c}')
    print()
    print('RAG search query:')
    print(f"  {two_pass.get('rag_search_query', 'N/A')}")
    print()
    print('Stratigraphic notes:')
    print(f"  {two_pass.get('stratigraphic_notes', 'N/A')}")
    print()
    print('Confidence zones:')
    for zone, desc in two_pass.get('confidence_zones', {}).items():
        print(f'  {zone}: {desc}')
    print()
    print('Restoration plan:')
    for i, item in enumerate(two_pass.get('restoration_plan', []), 1):
        print(f'  {i}. {item}')
    print()
    print('SDXL prompt component (sanitized):')
    print(f"  {two_pass.get('sdxl_prompt_component', 'N/A')}")

In [ ]:
# Cell 3: Stratigraphic confidence by folder
from collections import defaultdict

folder_scores = defaultdict(lambda: defaultdict(list))
for r in results:
    folder = r.get('source_folder', 'unknown')
    for key, val in r.get('stratigraphic_confidence', {}).items():
        if isinstance(val, (int, float)):
            folder_scores[folder][key].append(val)

conf_keys = ['confirmed_roman', 'roman_beneath_mosque', 'roman_adjacent', 'fully_reconstructed']
header = f"{'Folder':<35}" + "".join(f"{k[:14]:>15}" for k in conf_keys)
print(header)
print('-' * len(header))
for folder in sorted(folder_scores):
    row = f"{folder:<35}"
    for key in conf_keys:
        vals = folder_scores[folder].get(key, [])
        avg = sum(vals) / len(vals) if vals else 0.0
        row += f"{avg:>15.2f}"
    print(row)

In [ ]:
# Cell 4: Negative prompt
import sys
sys.path.insert(0, str(REPO_ROOT))
from src.vlm_analysis import build_negative_prompt

neg = build_negative_prompt(results)
print('NEGATIVE PROMPT:')
print()
print(neg)